In [10]:
from dash import Dash, html, dcc, Input, Output
import pandas as pd
import plotly.express as px

# تحميل البيانات
df = pd.read_csv('Reviews1.csv')

# تحضير البيانات
df['HelpfulnessRatio'] = df.apply(
    lambda row: row['HelpfulnessNumerator'] / row['HelpfulnessDenominator']
    if row['HelpfulnessDenominator'] > 0 else 0, axis=1)
df['ReviewLength'] = df['Text'].apply(lambda x: len(str(x).split()))
df['Time'] = pd.to_datetime(df['Time'], unit='s')
df['Year'] = df['Time'].dt.year

# إنشاء التطبيق
app = Dash(__name__, external_stylesheets=['https://codepen.io/chriddyp/pen/bWLwgP.css'])

# Layout
app.layout = html.Div([
    html.H1("📦 Enhanced Product Reviews Dashboard", style={'textAlign': 'center', 'color': 'white', 'backgroundColor': '#2c3e50'}),

    html.Div([
        html.Label("Select Score"),
        dcc.Dropdown(
            id='score-filter',
            options=[{'label': str(score), 'value': score} for score in sorted(df['Score'].unique())],
            placeholder="Select score"
        ),
        html.Label("Minimum Helpfulness Ratio"),
        dcc.Slider(
            id='helpfulness-slider',
            min=0,
            max=1,
            step=0.1,
            marks={0: '0%', 0.5: '50%', 1: '100%'},
            value=0
        )
    ], style={'marginBottom': '30px'}),

    html.Div(id='stats-box', style={'textAlign': 'center', 'backgroundColor': '#ecf0f1', 'padding': '10px', 'borderRadius': '10px'}),

    html.Hr(),

    html.Div([
        html.Div([dcc.Graph(id='score-distribution')], className='six columns'),
        html.Div([dcc.Graph(id='score-pie')], className='six columns')
    ], className='row'),

    html.Div([
        html.Div([dcc.Graph(id='top-users')], className='six columns'),
        html.Div([dcc.Graph(id='box-user-score')], className='six columns')
    ], className='row'),

    html.Div([
        html.Div([dcc.Graph(id='reviews-over-time')], className='six columns'),
        html.Div([dcc.Graph(id='review-length')], className='six columns')
    ], className='row'),

    html.H3("📝 Filtered Reviews Table"),
    html.Div(id='filtered-table')
])


@app.callback(
    [Output('score-distribution', 'figure'),
     Output('score-pie', 'figure'),
     Output('top-users', 'figure'),
     Output('box-user-score', 'figure'),
     Output('reviews-over-time', 'figure'),
     Output('review-length', 'figure'),
     Output('filtered-table', 'children'),
     Output('stats-box', 'children')],
    [Input('score-filter', 'value'),
     Input('helpfulness-slider', 'value')]
)
def update_dashboard(score, helpfulness_ratio):
    filtered = df.copy()

    if score is not None:
        filtered = filtered[filtered['Score'] == score]

    filtered = filtered[filtered['HelpfulnessRatio'] >= helpfulness_ratio]

    # رسم توزيع التقييمات
    fig1 = px.histogram(filtered, x='Score', nbins=10, title='Score Distribution')

    # Pie Chart للتقييمات
    fig2 = px.pie(filtered, names='Score', title='Score Distribution Pie Chart')

    # Top users حسب عدد المراجعات
    top_users = filtered['ProfileName'].value_counts().nlargest(10).reset_index()
    top_users.columns = ['ProfileName', 'ReviewCount']
    fig3 = px.bar(top_users, x='ProfileName', y='ReviewCount', title='Top 10 Active Reviewers')

    # Box plot: توزيع التقييمات حسب المستخدم
    fig4 = px.box(filtered[filtered['ProfileName'].isin(top_users['ProfileName'])],
                  x='ProfileName', y='Score', title='Score Variation for Top Users')

    # عدد المراجعات حسب السنة
    reviews_per_year = filtered.groupby('Year').size().reset_index(name='ReviewCount')
    fig5 = px.line(reviews_per_year, x='Year', y='ReviewCount', title='Reviews Over Time')

    # توزيع طول المراجعة
    fig6 = px.histogram(filtered, x='ReviewLength', nbins=30, title='Review Word Count Distribution')

    # جدول
    table = html.Table([
        html.Thead(html.Tr([html.Th(col) for col in ['ProfileName', 'Score', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Summary']])),
        html.Tbody([
            html.Tr([html.Td(row[col]) for col in ['ProfileName', 'Score', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Summary']])
            for _, row in filtered.head(10).iterrows()
        ])
    ])

    # إحصائيات
    total_reviews = len(filtered)
    avg_score = round(filtered['Score'].mean(), 2) if total_reviews > 0 else 0
    avg_helpful = round(filtered['HelpfulnessRatio'].mean() * 100, 2) if total_reviews > 0 else 0
    avg_length = round(filtered['ReviewLength'].mean(), 2) if total_reviews > 0 else 0

    stats = html.Div([
        html.H4(f"Total Reviews: {total_reviews}"),
        html.H4(f"Average Score: {avg_score}"),
        html.H4(f"Avg. Helpfulness: {avg_helpful}%"),
        html.H4(f"Avg. Review Length: {avg_length} words")
    ])

    return fig1, fig2, fig3, fig4, fig5, fig6, table, stats


if __name__ == '__main__':
    app.run(debug=True, port=8053)
